# Step 5: Analysis and Visualization

This notebook loads the Step 4 results and plots the retained optimization accuracy versus noise intensity for the standard and biological QAOA circuits.


In [ ]:
from collections import defaultdict
from pathlib import Path
import json

import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / "results" / "step4_results.json").exists():
    if (project_root.parent / "results" / "step4_results.json").exists():
        project_root = project_root.parent

results_path = project_root / "results" / "step4_results.json"
output_path = project_root / "results" / "step5_money_shot.png"

with results_path.open("r", encoding="utf-8") as file_handle:
    rows = json.load(file_handle)

grouped = defaultdict(list)
for row in rows:
    grouped[row["circuit_type"]].append(row)

for circuit_rows in grouped.values():
    circuit_rows.sort(key=lambda row: row["noise_scale"])

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)
fig.patch.set_facecolor("#f7f2ea")
ax.set_facecolor("#fffdf7")

palette = {"standard": "#c65d22", "biological": "#1f6f78"}
labels = {"standard": "Standard QAOA", "biological": "Biological QAOA"}

for circuit_type in ("standard", "biological"):
    circuit_rows = grouped[circuit_type]
    noise_levels = [row["noise_scale"] for row in circuit_rows]
    accuracies = [row["retained_accuracy"] for row in circuit_rows]
    ax.plot(
        noise_levels,
        accuracies,
        marker="o",
        linewidth=2.8,
        markersize=7,
        color=palette[circuit_type],
        label=labels[circuit_type],
    )

ax.set_title("QBio Step 5: Optimization Accuracy Under Noise", fontsize=15, pad=14, weight="bold")
ax.set_xlabel("Noise Intensity", fontsize=11)
ax.set_ylabel("Optimization Accuracy", fontsize=11)
ax.set_xticks(sorted({row["noise_scale"] for row in rows}))
ax.set_ylim(0.0, 1.05)
ax.grid(True, alpha=0.25, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="best")

fig.tight_layout()
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, bbox_inches="tight")
plt.show()
print(f"Saved plot to {output_path}")
